# Oficina de dados 2026.2 — Do que se morre e por que se interna no Brasil (DATASUS)

**Modelagem Informacional · FGV EMAp · Prof. Júlio César Chaves**
Ambiente: **Databricks Free Edition** (serverless, Unity Catalog) · Atividade **individual**

---

### Desafio

> Descobrir, **desde a data mais antiga até a mais recente**, qual a **doença que mais mata** e qual a **doença que mais causa internação hospitalar**, **ao longo dos anos**, em **cada estado brasileiro**.

Fontes: **SIM** (Sistema de Informações sobre Mortalidade) e **SIH/SUS** (Sistema de Informações Hospitalares), do DATASUS, com a **CID-10** como dicionário de doenças.

### A jornada

| # | Etapa | Pergunta que a etapa responde | Saída |
|---|---|---|---|
| 1 | **Find & Understand** | Que dado existe, desde quando, em que grão? O que é "doença" e "estado" nesse dado? | dicionário de dados + 4 decisões de modelagem |
| 2 | **Provision** | Como trago centenas de arquivos `.dbc` para o meu lake, de forma reexecutável? | Volume `bronze_raw` + tabelas `bronze_*` |
| 3 | **Prep** | Como harmonizo anos, códigos e UFs e monto um fato que responda às duas perguntas? | `silver_*`, `dim_cid`, `gold_fato_causa`, `gold_lideres` |
| 4 | **Analyze** | Quem lidera em cada UF e ano? Quando a liderança mudou? A resposta é robusta? | tabela UF × ano + mapa de líderes + leitura temporal |

Cada etapa é uma **linha da Matriz de Zachman** vista na aula 1: Find & Understand ≈ Análise de Requisitos e Modelo Conceitual; Provision ≈ Modelo Físico; Prep ≈ Modelo Lógico e Implementação; Analyze ≈ uso pelo negócio.

### Como este roteiro funciona

- O roteiro entrega **cerca de 70% do caminho pronto** (o *seed*). As células marcadas com **🧩 SUA VEZ** ficam para você: são os outros 30%. Em quase todas, a solução é **adaptar** algo que o seed já fez para o SIM.
- Comece no **modo `rapido`** (3 UFs pequenas, poucos anos): o pipeline inteiro roda em minutos. Quando tudo funcionar, mude para `completo` e rode de novo. Como cada célula é **idempotente**, só o que falta é baixado.
- Ao fim de cada etapa há uma caixa **Trilhas** com uma atividade por perfil. Faça **pelo menos uma por etapa**:
  **Prático** (executar, quebrar e consertar) · **Criativo** (narrativa, visual, manchete) · **Abstrato** (modelo, propriedade, "e se?").

### Referências

| Fonte | Onde entra |
|---|---|
| FTP DATASUS `ftp.datasus.gov.br/dissemin/publicos/` — `SIM/CID10/DORES`, `SIM/PRELIM/DORES`, `SIHSUS/200801_/dados` | microdados |
| `referencias/datasus/IT_SIHSUS_1603.pdf` — DATASUS, layout da AIH reduzida | colunas do SIH (1.3) |
| `referencias/microdatasus/` — SALDANHA, R., pacote microdatasus (MIT): `process_sim.R` e `process_sih.R` traduzidos para Python (`processa_sim.py`, `processa_sih.py`); livro *sis* (rfsaldanha.github.io/sis/sim.html) | significado dos códigos de cada coluna (1.3) |
| DATASUS — tabelas da **CID-10** (`www2.datasus.gov.br/cid10/V2008`) | dimensão de doenças |
| `referencias/anonimizacao/` — PEREIRA, G. M. C. (TCC EMAp, 2025) | k-anonimato na publicação |
| TypeSafe — primitivas do System One (docs.typesafe.ai/primitives) | Jev: `choice`, `score`, `noul` (4.6) |
| Databricks — função `ai_decide` (docs.databricks.com/aws/en/sql/language-manual/functions/ai_decide) | o mesmo julgamento em SQL (4.7) |

Os artigos e trabalhos citados estão na pasta **`referencias/`** do repositório; os links da web estão em `referencias/README.md`.

## 0. Preparação do ambiente

1. No Databricks Free Edition, traga o repositório da oficina, **`dl-trajetoria`**: **Workspace → Create → Git folder**, cole a URL do repositório e abra `notebooks/oficina_sim_temporal.ipynb`. Se preferir, use **Workspace → Import → File** com este `.ipynb`.
2. Conecte o notebook ao compute **Serverless**.
3. Rode as células de cima para baixo. As duas primeiras instalam as bibliotecas e **reiniciam o Python**. Depois do reinício, as variáveis anteriores se perdem; siga a partir da célula CONFIG.

> **Mensagens esperadas depois do `%pip install`.** Duas mensagens aparecem quase sempre e **não são erro**:
>
> - *"Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython()…"*: a célula seguinte já faz esse reinício.
> - *"Core Python package version(s) changed … idna: 3.11 -> 3.20 …"*: uma dependência das bibliotecas atualizou um pacote que o Databricks já trazia. Em geral nada quebra.
>
> **Se algo quebrar** (erro estranho de import, `display` ou PySpark), restaure o ambiente. No serverless, abra o painel **Environment** (à direita), em **Dependencies → Added** remova o pacote indicado na mensagem e clique em **Apply**. Depois rode de novo a partir da primeira célula. Como o notebook é idempotente, nada é baixado nem duplicado de novo.

> **Rede no Free Edition.** O DATASUS publica por **FTP**, e o serverless do Free Edition acessa esse FTP: o notebook lê e baixa tudo direto com `ftplib`. O FTP do DATASUS, porém, às vezes fica lento ou fora do ar. Se o download falhar, a célula mostra o **Plano B**: rode `python3 scripts/baixar_datasus.py --modo rapido` no seu computador ou no Codespaces e envie a pasta `volume/` gerada para o Volume (**Catalog → workspace → oficina_sus → Volumes → bronze_raw → Upload to this volume**, mantendo as subpastas, ou `databricks fs cp -r volume/ dbfs:/Volumes/workspace/oficina_sus/bronze_raw/`). No modo `rapido` são poucos MB.

> **Custo e cota.** O modo `completo` baixa ~2,5 GB do SIM e ~5 GB do SIH (4 meses por ano). Rode-o **uma vez**, depois de validar tudo no `rapido`.

In [ ]:
%pip install -q pyreaddbc typesafe-sdk

In [ ]:
# Reinicia o Python para carregar os pacotes recém-instalados (fica em célula separada do %pip).
dbutils.library.restartPython()

In [ ]:
# CONFIG ─ parâmetros do roteiro
import os, re, json, tempfile
from pyspark.sql import functions as F, Window

CATALOG = "workspace"        # catálogo padrão do Free Edition
SCHEMA  = "oficina_sus"
VOLUME  = "bronze_raw"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")
RAW = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"

def T(nome):
    "Nome totalmente qualificado de uma tabela da oficina."
    return f"{CATALOG}.{SCHEMA}.{nome}"

FTP = "ftp://ftp.datasus.gov.br/dissemin/publicos"

def ou_vazio(todo, construir, esquema):
    # Roda `construir()`. Se um TODO 🧩 ainda não foi resolvido, avisa e devolve uma tabela VAZIA
    # com o `esquema` certo: o notebook roda até o fim, e só aquela parte do resultado sai vazia.
    try:
        df = construir()
        df.schema          # força a análise agora (no serverless o Spark é preguiçoso)
        return df
    except Exception as e:
        print(f"⚠️  {todo} (ou um TODO anterior) ainda não resolvido: {type(e).__name__}: {str(e).splitlines()[0][:150]}")
        print("    Seguindo com resultado VAZIO para o notebook rodar até o fim.")
        return spark.createDataFrame([], esquema)

# Código IBGE da UF (2 primeiros dígitos do código de município) → sigla
UF_COD = {"11": "RO", "12": "AC", "13": "AM", "14": "RR", "15": "PA", "16": "AP", "17": "TO",
          "21": "MA", "22": "PI", "23": "CE", "24": "RN", "25": "PB", "26": "PE", "27": "AL", "28": "SE", "29": "BA",
          "31": "MG", "32": "ES", "33": "RJ", "35": "SP", "41": "PR", "42": "SC", "43": "RS",
          "50": "MS", "51": "MT", "52": "GO", "53": "DF"}

MODO = "rapido"   # "rapido": valida o pipeline em minutos · "completo": responde ao desafio
if MODO == "rapido":
    UFS            = ["AC", "AP", "RR"]
    ANOS_SIM       = [1996, 2008, 2019, 2020, 2021, 2024]
    ANOS_SIM_PRELIM = []
    ANOS_SIH       = [2008, 2019, 2020, 2021, 2025]
    MESES_SIH      = [1, 7]
else:
    UFS            = sorted(UF_COD.values())
    ANOS_SIM       = list(range(1996, 2025))   # definitivos (ver seção 1)
    ANOS_SIM_PRELIM = [2025]                   # preliminar: ano fechado mais recente
    ANOS_SIH       = list(range(2008, 2026))
    MESES_SIH      = [1, 4, 7, 10]             # 1 mês por trimestre (ver seção 4.5)
print(f"MODO={MODO} · {len(UFS)} UFs · lake bruto em {RAW}")

---
# 1. Find & Understand

## 1.1 Decompor a pergunta antes de procurar o dado

Cada palavra do desafio esconde uma **decisão de modelagem**. Responda primeiro no papel:

| Palavra do desafio | Pergunta que ela levanta | Onde decidimos |
|---|---|---|
| "doença" | Em que nível da CID-10? `I219` (subcategoria), `I21` infarto (categoria), `I20–I25` isquêmicas (grupo) ou capítulo IX circulatório? Parto é doença? Homicídio é doença? | 3.4 e 4.5 |
| "mais mata" | Que campo da declaração de óbito? Ela tem até 5 linhas de causas. | 1.2 (`CAUSABAS`, a **causa básica**) |
| "mais causa internação" | Contar AIH, pessoas ou dias? Toda AIH é uma internação nova? | 1.3 e 3.2 |
| "cada estado" | Estado onde a pessoa **mora** ou onde **morreu/foi internada**? | 3.2 |
| "ao longo dos anos" / "mais antiga até a mais recente" | Desde quando cada sistema existe no mesmo formato? Qual é o último ano **fechado**? | 1.2 |

## 1.2 Encontrar as fontes e a sua cobertura temporal

| Sistema | Pasta no FTP | Período | Classificação | Veredito |
|---|---|---|---|---|
| SIM (CID-9) | `SIM/CID9/DORES` | 1979–1995 | **CID-9** | ⚠️ Fora: outra classificação de doenças. Juntar as duas exige uma tabela de correspondência (trilha abstrata). |
| SIM | `SIM/CID10/DORES` | **1996 → 2024** | CID-10 | ✅ Dado **definitivo**, um arquivo por UF de **residência** e ano: `DO{UF}{AAAA}.dbc`. |
| SIM preliminar | `SIM/PRELIM/DORES` | 2025 → 2026 | CID-10 | ⚠️ Ainda muda. 2025 entra marcado como `preliminar`; 2026 está incompleto e fica fora. |
| SIH (layout antigo) | `SIHSUS/199201_200712` | 1992–2007 | CID-9 até 1997 | ⚠️ Fora: layout e classificação diferentes. |
| SIH | `SIHSUS/200801_/dados` | **2008-01 → mês mais recente** | CID-10 | ✅ AIH "reduzida", um arquivo por UF **do hospital** e **mês de competência**: `RD{UF}{AA}{MM}.dbc`. |
| CID-10 | `www2.datasus.gov.br/cid10/V2008/downloads/CID10CSV.zip` | versão 2008 | — | ✅ Categorias, grupos e capítulos. |

Os arquivos são **`.dbc`**: um DBF (formato dBase, de largura fixa) comprimido com um algoritmo próprio do DATASUS. Usaremos `pyreaddbc` para descomprimir e um leitor de DBF escrito aqui mesmo (seção 2). O caminho de linha de comando citado no syllabus (`dbf_dump`) só funciona **depois** de descomprimir.

### Checkpoint 1: entender antes de baixar

A célula abaixo **não baixa microdados**: só lista as pastas do FTP e calcula a cobertura.

In [ ]:
from ftplib import FTP as ConexaoFTP

def listar(pasta):
    with ConexaoFTP("ftp.datasus.gov.br", timeout=60) as ftp:
        ftp.login()
        return [os.path.basename(n) for n in ftp.nlst(f"/dissemin/publicos/{pasta}")]

try:
    sim = [n for n in listar("SIM/CID10/DORES") if re.match(r"DO[A-Z]{2}\d{4}\.dbc$", n, re.I)]
    anos = sorted({int(n[4:8]) for n in sim})
    print(f"SIM definitivo: {anos[0]} → {anos[-1]} · {len(sim)} arquivos")
    print("  'UFs' nos nomes:", sorted({n[2:4].upper() for n in sim}))
    prelim = [n for n in listar("SIM/PRELIM/DORES") if n.upper().startswith("DO")]
    print("SIM preliminar:", sorted({n[4:8] for n in prelim}))
    rd = [n for n in listar("SIHSUS/200801_/dados") if re.match(r"RD[A-Z]{2}\d{4}\.dbc$", n, re.I)]
    comp = sorted({"20" + n[4:8] for n in rd})
    print(f"SIH (RD): {comp[0][:4]}-{comp[0][4:]} → {comp[-1][:4]}-{comp[-1][4:]} · {len(rd)} arquivos")
except Exception as e:
    print("Falha ao acessar o FTP do DATASUS (instável? tente de novo em alguns minutos):", e)
    print("No Codespaces/terminal: curl --list-only ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/")
    print("No navegador (que não abre ftp://): https://datasus.saude.gov.br/transferencia-de-arquivos/")

> **Observe:** há **28** "UFs" nos nomes do SIM, não 27. A sobra é `BR`: o arquivo `DOBR{AAAA}.dbc` traz **o Brasil inteiro**, a mesma informação da soma das UFs. Lê-lo junto com as UFs **duplica** os óbitos. É a mesma armadilha do `perfil_eleitorado_BRASIL.csv` do TSE.
>
> Na pasta do SIH também há arquivos `RJ`, `SP` e `ER` (AIH rejeitadas, serviços profissionais, erros). Só o prefixo **`RD`** é a AIH reduzida. Cuidado: `RJ` aqui é tipo de arquivo, não o estado!

## 1.3 Dicionário de dados

Duas fontes, ambas na pasta `referencias/` do repositório:

- **Layout oficial** do DATASUS: `referencias/datasus/IT_SIHSUS_1603.pdf` (no FTP, em `SIHSUS/200801_/Doc/`). Lista **todas** as colunas da AIH reduzida, com tipo e descrição.
- **microdatasus** (SALDANHA, R.) em Python: `referencias/microdatasus/processa_sim.py` e `processa_sih.py`, traduzidos do pacote R original. O dicionário `CODIGOS` mostra o que cada código significa (ex.: `CODIGOS["SEXO"]` → `{'1': 'Masculino', ...}`). Abra o arquivo e leia, ou importe-o numa célula:

  ```python
  import sys; sys.path.append("../referencias/microdatasus")
  from processa_sih import CODIGOS
  CODIGOS["SEXO"]
  ```

  Colunas que o pacote não recodifica, como os diagnósticos, não estão em `CODIGOS`; procure-as no PDF. Há também `processar(df)`, que decodifica um `pandas.DataFrame` inteiro. É opcional e serve para explorar uma amostra.

**SIM** (preenchido pelo seed):

| Atributo do desafio | Coluna | Formato | Observação |
|---|---|---|---|
| doença que matou | `CAUSABAS` | CID-10 sem ponto, 3 ou 4 caracteres (`I219`, `I10X` ou `I10`) | **causa básica**: a doença que iniciou a cadeia de eventos |
| ano | nome do arquivo (`DO{UF}{AAAA}`) e `DTOBITO` (`ddmmaaaa`) | texto | |
| estado | `CODMUNRES` | município IBGE de residência: **7 dígitos até 2005, 6 depois** | os 2 primeiros dígitos são a UF |
| contexto | `SEXO`, `IDADE` | `IDADE` = unidade (1º dígito) + valor | não usados no desafio; úteis na trilha |

**SIH** — 🧩 **SUA VEZ (TODO 1.1):** preencha a tabela abaixo, no mesmo formato da tabela do SIM. Primeiro ache cada coluna no PDF `referencias/datasus/IT_SIHSUS_1603.pdf`: a de diagnóstico, a de ano (e mês) de competência, a de município de residência e a que identifica o **tipo da AIH**. Depois abra `referencias/microdatasus/processa_sih.py`, procure `'IDENT'` em `CODIGOS` e anote o significado de cada código. Você vai precisar dele na etapa 3.

| Atributo do desafio | Coluna | Formato | Observação |
|---|---|---|---|
| doença que causou a internação | | | |
| ano | | | |
| estado | | | |
| tipo de AIH | | | |

---
### Trilhas — Etapa 1

- **Prático:** baixe `DOAC2022.dbc` (óbitos do Acre em 2022) e abra o PDF `Estrutura_do_SIM_2025.pdf`. Confira o significado de `CAUSABAS` e de `CIRCOBITO`.
  - **PDF:** já está no repositório, em `referencias/datasus/Estrutura_do_SIM_2025.pdf`. No FTP fica em `ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DOCS/Estrutura_do_SIM_2025.pdf`.
  - **`.dbc` no próprio Databricks** (o FTP funciona daqui). Crie uma célula com:

    ```python
    from ftplib import FTP as ConexaoFTP
    destino = os.path.join(tempfile.gettempdir(), "DOAC2022.dbc")     # disco local do notebook
    with ConexaoFTP("ftp.datasus.gov.br", timeout=60) as ftp, open(destino, "wb") as f:
        ftp.login()
        ftp.retrbinary("RETR /dissemin/publicos/SIM/CID10/DORES/DOAC2022.dbc", f.write)
    print(destino, os.path.getsize(destino), "bytes")
    ```

    Na etapa 2, `pyreaddbc.dbc2dbf` e `ler_dbf` transformam esse arquivo em tabela.
  - **`.dbc` pelo navegador:** Chrome e Firefox não abrem mais links `ftp://`. Use a página do DATASUS **https://datasus.saude.gov.br/transferencia-de-arquivos/**: preencha **Fonte** = SIM, **Modalidade** = Dados, **Tipo de Arquivo** = DO (declarações de óbito), **Ano** = 2022 e **UF** = AC, e baixe o arquivo listado.
  - **`.dbc` pelo terminal** (Codespaces ou seu computador): `curl -O ftp://ftp.datasus.gov.br/dissemin/publicos/SIM/CID10/DORES/DOAC2022.dbc`.
  - Para abrir o `.dbc` fora do Databricks: `pip install pyreaddbc` e `pyreaddbc.dbc2dbf("DOAC2022.dbc", "DOAC2022.dbf")`. O DBF abre no LibreOffice Calc ou com a função `ler_dbf` da etapa 2.
- **Criativo:** escreva, em 3 linhas, a "persona" de um gestor da Secretaria Estadual de Saúde que usaria esta tabela. Que decisão de orçamento ele tomaria com ela?
- **Abstrato:** o SIM organiza os arquivos por UF de **residência** e o SIH por UF **do hospital**. Em qual coluna de Zachman (o quê, como, onde, quem, quando, por quê) mora essa diferença? Que pergunta de negócio cada escolha responde melhor?

---
# 2. Provision (camada bronze)

Princípio: **o bronze guarda o dado como veio**, mais metadados de linhagem. Aqui o arquivo `.dbc` original fica intacto no Volume. A tabela bronze leva **só as colunas que a pergunta usa**, ainda como texto e sem nenhuma limpeza. Se precisarmos de outra coluna depois, reprocessamos a partir do `.dbc`.

## 2.1 Baixar de forma idempotente

Uma operação é **idempotente** quando rodá-la 1 ou N vezes deixa o lake no mesmo estado: f(f(x)) = f(x). É o que permite reexecutar uma célula que falhou sem pensar no que já rodou. No Spark, `overwrite` e `MERGE INTO` são idempotentes; `append` e `INSERT INTO` não são, porque cada execução duplica as linhas.

A função abaixo pula o arquivo que já está no Volume. Mas "já está" só é verdade se o arquivo estiver **completo**. Por isso a cópia vai primeiro para `destino.part` e só no fim é **renomeada** para `destino`. Se a cópia cair no meio, sobra um `.part` (apagado no `finally`), e a próxima execução baixa o arquivo de novo.

In [ ]:
import shutil, urllib.request

def baixar(url, destino):
    "Baixa `url` para `destino` (caminho no Volume). Não baixa de novo se já existe."
    if os.path.exists(destino) and os.path.getsize(destino) > 0:
        return "já no lake"
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    tmp = os.path.join(tempfile.gettempdir(), os.path.basename(destino) + ".part")
    parcial = destino + ".part"
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "oficina-emap/2026"})
        with urllib.request.urlopen(req, timeout=300) as r, open(tmp, "wb") as f:
            shutil.copyfileobj(r, f, 1 << 20)
        shutil.copyfile(tmp, parcial)       # baixa no disco local, depois copia para o Volume
        os.replace(parcial, destino)        # renomear é atômico: `destino` nunca fica pela metade
        return "baixado"
    except Exception as e:
        return f"FALHOU ({e})"
    finally:
        for p in (tmp, parcial):
            if os.path.exists(p):
                os.remove(p)

## 2.2 Ler DBF: um formato de **largura fixa**

Um DBF é um cabeçalho (nome, tipo e largura de cada campo) seguido de registros **todos do mesmo tamanho**, um atrás do outro. É um formato **orientado a linha**: para ler uma coluna, você precisa passar por todas as linhas. Por isso o convertemos para **Parquet**, que é **colunar**.

A largura fixa tem uma vantagem: o arquivo inteiro vira uma matriz de bytes (`numpy`), e cada coluna é só uma fatia dela. Sem laços em Python, um ano inteiro de SP é lido em segundos.

In [ ]:
import struct, numpy as np, pandas as pd, pyreaddbc

def ler_dbf(caminho, colunas):
    "Lê só `colunas` de um DBF como texto. Colunas ausentes (layout antigo) voltam vazias."
    with open(caminho, "rb") as f:
        n_reg, tam_cab, tam_reg = struct.unpack("<IHH", f.read(32)[4:12])
        campos, pos = {}, 1                       # byte 0 de cada registro = marca de apagado
        while (d := f.read(32))[0] != 0x0D:       # descritores de 32 bytes até o terminador 0x0D
            nome = d[:11].split(b"\0")[0].decode("latin-1").upper()
            campos[nome] = (pos, d[16])           # (posição, largura)
            pos += d[16]
        f.seek(tam_cab)
        bruto = np.frombuffer(f.read(n_reg * tam_reg), dtype=np.uint8)
    bruto = bruto[: len(bruto) // tam_reg * tam_reg].reshape(-1, tam_reg)
    bruto = bruto[bruto[:, 0] != ord("*")]        # descarta registros apagados
    df = {}
    for c in colunas:
        if c in campos:
            p, w = campos[c]
            fatia = np.ascontiguousarray(bruto[:, p:p + w]).view(f"S{w}").ravel()
            df[c] = pd.Series(fatia).str.decode("latin-1").str.strip()
        else:
            df[c] = pd.Series([None] * len(bruto), dtype="string")
    return pd.DataFrame(df).astype("string")

def dbc_para_parquet(sistema, nome_dbc, colunas):
    "Volume: dbc/{sistema}/X.dbc  →  parquet/{sistema}/X.parquet (só `colunas`)."
    origem  = f"{RAW}/dbc/{sistema}/{nome_dbc}"
    destino = f"{RAW}/parquet/{sistema}/{nome_dbc[:-4]}.parquet"
    if os.path.exists(destino):
        return
    local = os.path.join(tempfile.gettempdir(), nome_dbc[:-4])
    try:
        pyreaddbc.dbc2dbf(origem, local + ".dbf")
        ler_dbf(local + ".dbf", colunas).to_parquet(local + ".parquet", index=False)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        shutil.copyfile(local + ".parquet", destino)
    finally:
        for ext in (".dbf", ".parquet"):
            if os.path.exists(local + ext):
                os.remove(local + ext)

## 2.3 Orquestrar: baixar + converter, em paralelo

`provisionar` recebe uma lista de nomes de arquivo e faz as duas etapas para cada um. Ao final, mostra o **Plano B** para o que falhou.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

def provisionar(sistema, pasta_ftp, nomes, colunas, paralelo=4):
    def um(nome):
        st = baixar(f"{FTP}/{pasta_ftp}/{nome}", f"{RAW}/dbc/{sistema}/{nome}")
        if not st.startswith("FALHOU"):
            dbc_para_parquet(sistema, nome, colunas)
        return nome, st
    with ThreadPoolExecutor(paralelo) as ex:
        res = list(ex.map(um, nomes))
    falhas = [(n, s) for n, s in res if s.startswith("FALHOU")]
    print(f"{sistema}: {len(res) - len(falhas)}/{len(res)} arquivos no lake")
    for n, s in falhas[:10]:
        print(f"  ✗ {n}: {s}")
    if falhas:
        print(f"  PLANO B → baixe de {FTP}/{pasta_ftp}/ e envie para {RAW}/dbc/{sistema}/ ; depois rode de novo.")
    return res

COLUNAS_SIM = ["DTOBITO", "CAUSABAS", "CODMUNRES", "SEXO", "IDADE"]
NOMES_SIM   = [f"DO{uf}{ano}.dbc" for uf in UFS for ano in ANOS_SIM]          # nunca DOBR!
_ = provisionar("sim", "SIM/CID10/DORES", NOMES_SIM, COLUNAS_SIM)
if ANOS_SIM_PRELIM:
    _ = provisionar("sim_prelim", "SIM/PRELIM/DORES",
                    [f"DO{uf}{ano}.dbc" for uf in UFS for ano in ANOS_SIM_PRELIM], COLUNAS_SIM)

In [ ]:
# 🧩 SUA VEZ (TODO 2.1) — provisionar o SIH
# Os arquivos seguem o padrão RD{UF}{AA}{MM}.dbc, ex.: RDAC2401.dbc = Acre, jan/2024.
# Use UFS, ANOS_SIH e MESES_SIH. Atenção ao ano com 2 dígitos.
NOMES_SIH   = ...   # lista de nomes de arquivo
COLUNAS_SIH = ...   # as colunas do seu dicionário (TODO 1.1); inclua também "MES_CMPT"

if NOMES_SIH is Ellipsis or COLUNAS_SIH is Ellipsis:
    print("⚠️  TODO 2.1 ainda não resolvido: o SIH não foi provisionado. O notebook segue só com o SIM.")
else:
    _ = provisionar("sih", "SIHSUS/200801_/dados", NOMES_SIH, COLUNAS_SIH)

## 2.4 CID-10 e tabelas bronze

In [ ]:
import zipfile

st = baixar("http://www2.datasus.gov.br/cid10/V2008/downloads/CID10CSV.zip", f"{RAW}/cid/CID10CSV.zip")
print("CID10CSV.zip:", st)
with zipfile.ZipFile(f"{RAW}/cid/CID10CSV.zip") as z:
    for arq, tabela in [("CID-10-CAPITULOS.CSV", "bronze_cid_capitulos"),
                        ("CID-10-GRUPOS.CSV", "bronze_cid_grupos"),
                        ("CID-10-CATEGORIAS.CSV", "bronze_cid_categorias")]:
        cid = pd.read_csv(z.open(arq), sep=";", encoding="latin-1", dtype=str)
        cid = cid.loc[:, ~cid.columns.str.startswith("Unnamed")]       # ';' no fim de cada linha
        spark.createDataFrame(cid.fillna("")).write.mode("overwrite") \
             .option("overwriteSchema", True).saveAsTable(T(tabela))

def bronze(sistema, tabela):
    pasta = f"{RAW}/parquet/{sistema}"
    if not os.path.isdir(pasta) or not os.listdir(pasta):
        print(f"– {tabela}: nada em {pasta}")
        return
    (spark.read.option("mergeSchema", True).parquet(pasta)
          .withColumn("_arquivo", F.col("_metadata.file_name"))
          .withColumn("_ingestao", F.current_timestamp())
          .write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T(tabela)))
    print(f"✓ {tabela}: {spark.table(T(tabela)).count():,} linhas")

bronze("sim", "bronze_sim")
bronze("sim_prelim", "bronze_sim_prelim")
bronze("sih", "bronze_sih")

---
### Trilhas — Etapa 2

- **Prático:** rode de novo todas as células da seção 2 e confirme que nada é baixado outra vez. Depois apague um `.parquet` do Volume e rode de novo: só ele é refeito.
- **Criativo:** desenhe o "mapa do lake" (FTP → `dbc/` → `parquet/` → `bronze_*` → `silver_*` → `gold_*`) como se fosse outro lugar onde coisas chegam, são guardadas e transformadas. Por exemplo, um **porto marítimo**: o navio (FTP) traz **contêineres lacrados** (`.dbc`), que são descarregados no cais (`dbc/`), abertos e reorganizados em paletes (`parquet/`), registrados na alfândega como chegaram (`bronze_*`), conferidos e separados por destino (`silver_*`) e despachados prontos para a loja (`gold_*`). Outras ideias: uma **cozinha** (feira → despensa → pré-preparo → prato) ou os **Correios** (coleta → triagem → centro de distribuição → entrega). O que **não** cabe na sua metáfora também é achado: onde ela quebra?
- **Abstrato:** compare o custo de ler **uma coluna** de 1 milhão de óbitos em DBF (linha, largura fixa) e em Parquet (colunar, comprimido). Por que o leitor `ler_dbf` acima consegue ser rápido mesmo sendo um formato de linha? Que propriedade do formato ele explora?

---
# 3. Prep (camadas silver e gold)

## 3.1 Conhecer o bronze antes de transformar

In [ ]:
display(spark.table(T("bronze_sim"))
        .groupBy(F.substring("_arquivo", 5, 4).alias("ano_arquivo"))
        .agg(F.count("*").alias("obitos"),
             F.round(F.avg(F.length("CODMUNRES")), 2).alias("digitos_CODMUNRES"),
             F.sum((F.length("CAUSABAS") == 3).cast("int")).alias("causabas_3_chars"),
             F.sum(F.col("CAUSABAS").endswith("X").cast("int")).alias("causabas_terminada_em_X"),
             F.sum((F.substring("DTOBITO", 5, 4) != F.substring("_arquivo", 5, 4)).cast("int")).alias("ano_dtobito_difere"))
        .orderBy("ano_arquivo"))

> **Leitura.** O código de município muda de 7 para 6 dígitos ao longo da série, e uma mesma categoria aparece como `I10`, `I10X` ou `I109`. Os **3 primeiros caracteres** da CID e os **2 primeiros** do município são estáveis: é neles que harmonizamos.

## 3.2 Silver: um registro por óbito e por internação, com chaves harmonizadas

Decisões do seed:
- **estado = UF de residência** (2 primeiros dígitos do município). Código ignorado → `uf` nula, contada e excluída.
- **doença = categoria CID-10** (3 caracteres). Revisitamos isso na seção 4.5.
- **ano do óbito = ano do arquivo** (o SIM organiza os arquivos por ano do óbito).

In [ ]:
uf_de = F.create_map(*[F.lit(x) for kv in UF_COD.items() for x in kv])

def silver_sim(tabela, preliminar):
    return (spark.table(T(tabela))
            .select(F.substring("_arquivo", 5, 4).cast("int").alias("ano"),
                    uf_de[F.substring("CODMUNRES", 1, 2)].alias("uf"),
                    F.upper(F.substring("CAUSABAS", 1, 3)).alias("cid_cat"),
                    F.lit(preliminar).alias("preliminar"), "_arquivo"))

obitos = silver_sim("bronze_sim", False)
if spark.catalog.tableExists(T("bronze_sim_prelim")) and ANOS_SIM_PRELIM:
    obitos = obitos.unionByName(silver_sim("bronze_sim_prelim", True))
obitos.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T("silver_obitos"))
ESQUEMA_INTERNACOES = "ano INT, uf STRING, cid_cat STRING, mes INT, _arquivo STRING"   # o formato do TODO 3.1

display(spark.table(T("silver_obitos")).groupBy("preliminar")
        .agg(F.count("*").alias("obitos"), F.sum(F.col("uf").isNull().cast("int")).alias("uf_ignorada"),
             F.min("ano").alias("de"), F.max("ano").alias("ate")))

In [ ]:
# 🧩 SUA VEZ (TODO 3.1) — silver_internacoes
# Mesmo formato de silver_obitos: ano, uf (de RESIDÊNCIA), cid_cat, mes, _arquivo.
# Lembre do seu dicionário (TODO 1.1): qual valor de IDENT é uma internação NOVA?
def construir_internacoes():
    return (spark.table(T("bronze_sih"))
            .where(...)                          # filtro do tipo de AIH
            .select(...))                        # ano, uf, cid_cat, mes, _arquivo

internacoes = ou_vazio("TODO 3.1", construir_internacoes, ESQUEMA_INTERNACOES)
internacoes.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T("silver_internacoes"))
display(spark.table(T("silver_internacoes")).groupBy("ano").count().orderBy("ano"))

## 3.3 Dimensão de doenças (`dim_cid`)

A CID-10 tem uma hierarquia **capítulo → grupo → categoria**. Os capítulos e grupos vêm como **intervalos** de categorias (`A00`–`B99`), então a junção é por faixa (`BETWEEN`), não por igualdade. A comparação de texto funciona porque todos os códigos têm o formato letra + 2 dígitos.

In [ ]:
cat = spark.table(T("bronze_cid_categorias")).select(F.col("CAT").alias("cid_cat"), F.col("DESCRICAO").alias("descricao"))
cap = spark.table(T("bronze_cid_capitulos")).select(F.col("NUMCAP").cast("int").alias("capitulo"),
                                                    F.col("CATINIC").alias("c_ini"), F.col("CATFIM").alias("c_fim"),
                                                    F.col("DESCRABREV").alias("capitulo_desc"))
grp = spark.table(T("bronze_cid_grupos")).select(F.col("CATINIC").alias("g_ini"), F.col("CATFIM").alias("g_fim"),
                                                 F.col("DESCRICAO").alias("grupo_desc"))
juncao = (cat.join(cap, F.col("cid_cat").between(F.col("c_ini"), F.col("c_fim")), "left")
              .join(grp, F.col("cid_cat").between(F.col("g_ini"), F.col("g_fim")), "left"))
print("categorias:", cat.count(), "· linhas depois da junção por faixa:", juncao.count())
display(juncao.groupBy("cid_cat").count().where("count > 1")
        .join(juncao, "cid_cat").select("cid_cat", "g_ini", "g_fim", "grupo_desc").orderBy("cid_cat", "g_ini").limit(9))

> **Armadilha: a junção por faixa duplicou linhas.** Os grupos de neoplasias são **aninhados**: `C00–C97` (neoplasias malignas) contém `C00–C75`, que contém `C00–C14` (lábio, cavidade oral e faringe). A tabela de grupos é, na verdade, uma **hierarquia**, e não uma partição. Decisão do seed: cada categoria fica com o grupo **mais específico**, o de início mais tardio e fim mais cedo.

> **Armadilha semântica: COVID-19.** No Brasil, a COVID-19 foi codificada como **`B34.2`** ("infecção por coronavírus de localização não especificada"), e não como `U07.1`. Na categoria de 3 caracteres ela aparece como **`B34` — "Doença por vírus, de localização não especificada"**. A descrição oficial esconde a doença que mais matou em 2020–2021. Nenhuma regra sobre o texto da CID descobriria isso: é **conhecimento de domínio** que precisa entrar no modelo. Por isso a `dim_cid` ganha um atributo `observacao`.

In [ ]:
OBSERVACOES = {"B34": "Em 2020–2022 dominada por B34.2 = COVID-19 (codificação adotada no Brasil)"}
obs = spark.createDataFrame(list(OBSERVACOES.items()), ["cid_cat", "observacao"])

mais_especifico = Window.partitionBy("cid_cat").orderBy(F.desc_nulls_last("g_ini"), F.asc("g_fim"))
dim_cid = (juncao.withColumn("_r", F.row_number().over(mais_especifico)).where("_r = 1")
           .join(obs, "cid_cat", "left")
           .select("cid_cat", "descricao", "capitulo", "capitulo_desc",
                   F.concat_ws("-", "g_ini", "g_fim").alias("grupo"), "grupo_desc", "observacao"))
dim_cid.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T("dim_cid"))
dim_cid = spark.table(T("dim_cid"))
print("dim_cid:", dim_cid.count(), "linhas · distintas:", dim_cid.select("cid_cat").distinct().count())

# Códigos usados nos microdados que NÃO estão na tabela da CID
usados = spark.table(T("silver_obitos")).select("cid_cat").unionByName(spark.table(T("silver_internacoes")).select("cid_cat"))
display(usados.join(dim_cid, "cid_cat", "left_anti").groupBy("cid_cat").count().orderBy(F.desc("count")))

> **Confira:** agora "linhas" e "distintas" são iguais. Se aparecer algum código órfão acima (sem descrição na CID de 2008, como códigos `U` criados depois), ele fica fora da `dim_cid` e com `capitulo` nulo no fato: documente-o.

## 3.4 O que conta como "doença"? (decisão de modelagem)

Sem filtro, a "doença" que mais interna no Brasil é o **parto normal** (`O80`), e entre as que mais matam aparece **`R99` — "outras causas mal definidas"**. Nenhuma das duas responde ao espírito da pergunta. O seed exclui cinco capítulos. Em vez de apagar essas linhas, guardamos tudo no gold e **filtramos na consulta**: assim a decisão pode ser revista sem reprocessar nada.

In [ ]:
CAPITULOS_FORA = {
    15: "Gravidez, parto e puerpério (O)",
    18: "Sintomas e achados anormais: causas mal definidas (R)",
    19: "Lesões e envenenamentos (S–T): a natureza da lesão, não uma doença",
    20: "Causas externas (V–Y): acidentes, agressões, suicídios",
    21: "Fatores que influenciam o contato com serviços de saúde (Z)",
}
display(dim_cid.where(F.col("capitulo").isin(list(CAPITULOS_FORA))).groupBy("capitulo", "capitulo_desc").count().orderBy("capitulo"))

## 3.5 Gold: um fato para as duas perguntas

Grão: **quantidade de eventos de uma fonte (óbito ou internação), em um ano, em uma UF de residência, por categoria CID**. As duas perguntas do desafio viram **dois valores de `fonte`** na mesma tabela, como o formato longo do roteiro do IPG.

Só entram as UFs carregadas (`UFS`). No modo `rapido`, os arquivos do SIH do Acre trazem também pacientes que **moram** no Amazonas. Mas os amazonenses internados em Manaus estão em `RDAM…`, que não baixamos. Contar o AM a partir dos arquivos do AC daria um número parcial e enganoso.

In [ ]:
def contagem(silver, fonte):
    return (spark.table(T(silver)).where(F.col("uf").isin(UFS))
            .groupBy(F.lit(fonte).alias("fonte"), "ano", "uf", "cid_cat")
            .agg(F.count("*").alias("n")))

In [ ]:
# 🧩 SUA VEZ (TODO 3.2) — juntar as duas fontes no mesmo fato
fato = contagem("silver_obitos", "obito")   # ... e as internações?

fato = fato.join(dim_cid.select("cid_cat", "capitulo"), "cid_cat", "left")
fato.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T("gold_fato_causa"))
display(spark.table(T("gold_fato_causa")).groupBy("fonte").agg(F.sum("n").alias("eventos"), F.min("ano"), F.max("ano")))

### Ranking: a doença líder em cada UF e ano

`lideres` aplica a decisão do 3.4, permite mudar o **nível** da CID (`cid_cat`, `grupo` ou `capitulo`) e numera as causas por UF e ano com uma **função de janela**.

In [ ]:
def lideres(fonte, nivel="cid_cat", fora=CAPITULOS_FORA, top=3):
    f = (spark.table(T("gold_fato_causa")).where(F.col("fonte") == fonte)
         .where(~F.coalesce(F.col("capitulo").isin(list(fora) or [-1]), F.lit(False))))
    if nivel != "cid_cat":
        f = f.drop("capitulo").join(dim_cid.select("cid_cat", nivel), "cid_cat", "left")
    g = f.groupBy("ano", "uf", nivel).agg(F.sum("n").alias("n"))
    w = Window.partitionBy("ano", "uf")
    return (g.withColumn("pct", F.round(F.col("n") / F.sum("n").over(w), 4))
             .withColumn("pos", F.row_number().over(w.orderBy(F.desc("n"), F.asc(nivel))))
             .where(F.col("pos") <= top)
             .withColumn("fonte", F.lit(fonte)))

gold_lideres = (lideres("obito").unionByName(lideres("internacao"))
                .join(dim_cid.select("cid_cat", "descricao", "observacao"), "cid_cat", "left"))
gold_lideres.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T("gold_lideres"))
display(spark.table(T("gold_lideres")).where("pos = 1").orderBy("fonte", "uf", "ano").limit(30))

## 3.6 Governança: publicar sem expor pessoas

Os microdados do SIM e do SIH não têm nome nem CPF, mas a combinação **município × ano × causa rara × sexo × idade** pode apontar uma pessoa, sobretudo em municípios pequenos. Esses campos são os *quasi-identificadores* do TCC de Pereira. O nosso `gold` agrega por **UF**, e o risco cai muito. Mesmo assim, aplicamos a regra da oficina: **nenhuma tabela publicada tem célula com menos de *k* = 5 eventos**.

In [ ]:
K = 5
fato = spark.table(T("gold_fato_causa"))
display(fato.agg(F.count("*").alias("celulas"),
                 F.sum((F.col("n") < K).cast("int")).alias(f"celulas_n_menor_{K}"))
            .withColumn("pct", F.round(F.col(f"celulas_n_menor_{K}") / F.col("celulas"), 3)))
fato.where(F.col("n") >= K).write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T("gold_fato_causa_publico"))

---
### Trilhas — Etapa 3

- **Prático:** troque o filtro `IDENT == "1"` por nenhum filtro. Quantas internações a mais aparecem? A líder de alguma UF muda?
- **Criativo:** escolha um óbito fictício de 2005 em um município de 7 dígitos e conte a "biografia" dele, do `.dbc` até a `gold_lideres`: em que linha de código cada atributo dele foi transformado?
- **Abstrato:** `gold_fato_causa` guarda **todos** os capítulos, e o filtro do 3.4 é aplicado na consulta. Compare com filtrar já no silver. Que pergunta futura cada escolha permite ou impede? Relacione com *schema-on-read* × *schema-on-write*.

---
# 4. Analyze

## 4.1 A doença que mais mata, por UF e ano

In [ ]:
def tabela_lideres(fonte):
    return (spark.table(T("gold_lideres")).where((F.col("fonte") == fonte) & (F.col("pos") == 1))
            .groupBy("uf").pivot("ano").agg(F.first("cid_cat")).orderBy("uf"))

display(tabela_lideres("obito"))

## 4.2 A doença que mais causa internação, por UF e ano

In [ ]:
# 🧩 SUA VEZ (TODO 4.1) — a mesma tabela para as internações
...

## 4.3 Mapa de líderes

Uma linha por UF e uma coluna por ano. Cada célula mostra o código da doença líder. As **três** doenças que mais lideram ganham cor e as demais ficam em cinza: com mais cores, o olho não distingue mais as categorias. O código escrito dentro da célula garante que a leitura não dependa só da cor.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

CORES = ["#2a78d6", "#eb6834", "#1baf7a"]          # 3 cores categóricas validadas para leitura daltônica
OUTRA, FUNDO, TINTA, TINTA2 = "#e6e5e0", "#fcfcfb", "#0b0b0b", "#52514e"
ROTULO_CURTO = {"B34": "COVID-19 (codificada como B34.2; ver dim_cid.observacao)"}

def mapa_lideres(fonte, titulo):
    lid = spark.table(T("gold_lideres")).where((F.col("fonte") == fonte) & (F.col("pos") == 1)).toPandas()
    grade = lid.pivot(index="uf", columns="ano", values="cid_cat").sort_index(ascending=False)
    frequentes = lid["cid_cat"].value_counts().index[:3].tolist()
    cor = {c: CORES[i] for i, c in enumerate(frequentes)}
    desc = lid.drop_duplicates("cid_cat").set_index("cid_cat")["descricao"].to_dict()

    fig, ax = plt.subplots(figsize=(max(9, 0.45 * grade.shape[1] + 2), 0.32 * grade.shape[0] + 1.6), facecolor=FUNDO)
    ax.set_facecolor(FUNDO)
    for i, uf in enumerate(grade.index):
        for j, ano in enumerate(grade.columns):
            c = grade.loc[uf, ano]
            if isinstance(c, str):
                ax.add_patch(plt.Rectangle((j, i), 1, 1, facecolor=cor.get(c, OUTRA), edgecolor=FUNDO, linewidth=2))
                ax.text(j + 0.5, i + 0.5, c, ha="center", va="center", fontsize=6, color=TINTA)
    ax.set_xlim(0, grade.shape[1]); ax.set_ylim(0, grade.shape[0])
    ax.xaxis.tick_top()
    ax.set_xticks([j + 0.5 for j in range(grade.shape[1])], grade.columns, rotation=90, fontsize=7, color=TINTA2)
    ax.set_yticks([i + 0.5 for i in range(grade.shape[0])], grade.index, fontsize=7, color=TINTA2)
    ax.tick_params(length=0)
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(titulo, loc="left", fontsize=11, color=TINTA, pad=28)
    leg = [Patch(facecolor=cor[c], label=f"{c} — {ROTULO_CURTO.get(c) or desc.get(c) or 's/ descrição'}") for c in frequentes]
    leg.append(Patch(facecolor=OUTRA, label="outras"))
    ax.legend(handles=leg, loc="upper left", bbox_to_anchor=(0, -0.02), ncol=1, frameon=False, fontsize=8, labelcolor=TINTA2)
    plt.tight_layout()
    plt.show()

mapa_lideres("obito", "Doença que mais mata, por UF de residência e ano (categoria CID-10)")

In [ ]:
# 🧩 SUA VEZ (TODO 4.2) — o mapa das internações
...

## 4.4 Da mais antiga à mais recente: quando a liderança mudou?

Para cada UF e fonte: a líder no **primeiro** ano, a líder no **último**, quantas vezes a liderança trocou de mãos e em quais anos a `B34` (COVID-19) liderou.

In [ ]:
lid = spark.table(T("gold_lideres")).where("pos = 1")
w = Window.partitionBy("fonte", "uf").orderBy("ano")
trajetoria = (lid.withColumn("anterior", F.lag("cid_cat").over(w))
                 .groupBy("fonte", "uf")
                 .agg(F.min("ano").alias("de"), F.max("ano").alias("ate"),
                      F.min_by("cid_cat", "ano").alias("lider_inicio"),
                      F.max_by("cid_cat", "ano").alias("lider_fim"),
                      F.sum((F.col("anterior").isNotNull() & (F.col("anterior") != F.col("cid_cat"))).cast("int")).alias("trocas"),
                      F.array_sort(F.collect_list(F.when(F.col("cid_cat") == "B34", F.col("ano")))).alias("anos_B34")))
trajetoria.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(T("gold_trajetoria"))
display(spark.table(T("gold_trajetoria")).orderBy("fonte", "uf"))

🧩 **SUA VEZ (TODO 4.3) — a resposta do desafio, em texto.** Com a tabela acima e os dois mapas, escreva até 6 frases:

1. Qual doença mais mata **na maioria das UFs** no início e no fim da série? Há exceções regionais?
2. O que aconteceu em 2020–2021? Em quantas UFs a `B34` liderou e por quanto tempo?
3. Qual doença mais causa internação? Ela é a mesma que mais mata? Por que faz sentido que não seja?

*(escreva aqui)*

## 4.5 O resultado é robusto?

Um resultado só vira **insight** se sobreviver a decisões de modelagem razoáveis. A célula abaixo recalcula a líder sob outras escolhas e mede **em quantos pares (UF, ano) a líder continua a mesma**.

In [ ]:
def concordancia(fonte, **cenario):
    base = lideres(fonte, top=1).select("ano", "uf", F.col("cid_cat").alias("base"))
    nivel = cenario.get("nivel", "cid_cat")
    alt = lideres(fonte, top=1, **cenario).select("ano", "uf", F.col(nivel).alias("alt"))
    if nivel != "cid_cat":     # compara no mesmo nível: leva a líder-base para o nível alternativo
        base = base.join(dim_cid.select(F.col("cid_cat").alias("base"), F.col(nivel).alias("base_nivel")), "base", "left") \
                   .drop("base").withColumnRenamed("base_nivel", "base")
    j = base.join(alt, ["ano", "uf"])
    return j.agg(F.round(F.avg((F.col("base") == F.col("alt")).cast("int")), 3)).first()[0]

cenarios = {
    "sem excluir capítulos (parto, mal definidas, causas externas entram)": dict(fora={}),
    "excluindo só gravidez/parto e mal definidas": dict(fora={15: "", 18: ""}),
    "doença = grupo CID (ex.: I20–I25 isquêmicas)": dict(nivel="grupo"),
    "doença = capítulo CID (ex.: IX circulatório)": dict(nivel="capitulo"),
}
display(pd.DataFrame([{"cenário": nome,
                       "óbitos: mesma líder": concordancia("obito", **kw),
                       "internações: mesma líder": concordancia("internacao", **kw)}
                      for nome, kw in cenarios.items()]))

> **Como ler.** 1,0 = a decisão não muda nada; 0,3 = a resposta depende quase toda da decisão. No cenário de **grupo** e de **capítulo**, comparamos a líder do seed "subida" para o mesmo nível: se ela continua líder, há concordância. Uma concordância baixa **não é erro**: é a prova de que a sua decisão de modelagem precisa aparecer no relatório.
>
> Há uma quinta decisão que esta célula não testa: o **mês** do SIH. No modo `completo` usamos 4 meses por ano (um por trimestre). Doenças respiratórias são sazonais. Se só janeiro fosse usado, a pneumonia de inverno sumiria. Teste isso na trilha prática abaixo.

## 4.6 (Opcional) Um novo atributo na `dim_cid` com o Jev

A regra do 3.4 classifica por **capítulo**, de forma determinística. Ela não sabe, por exemplo, que `B34` em 2021 é COVID-19, nem que `Z51` são sessões de quimioterapia. Este é o padrão da **A2**: do texto (a descrição da doença) nasce um **atributo tipado em uma dimensão existente**.

Há dois caminhos para o mesmo **modelo de decisão** (um modelo que não gera texto: devolve julgamentos tipados com probabilidades). Os dois são **opcionais** e aparecem aqui de forma superficial; faça um, outro ou nenhum:

| | 4.6 — TypeSafe (Jev) | 4.7 — Databricks `ai_decide` |
|---|---|---|
| Onde roda | Python, via `typesafe-sdk` (aqui, no Codespaces ou em qualquer lugar) | SQL, dentro do Databricks |
| Chave | sua chave TypeSafe, num *secret scope* | nenhuma: usa a governança do workspace |
| Perguntas | `Choice`, `Score`, `Noul` | o mesmo JSON: `type` = `choice`, `score`, `noul` |
| Status | API estável; pinar a versão do modelo | **Beta**, só em algumas regiões; pode não existir no seu workspace |

Para cada categoria que já apareceu entre as 3 primeiras (`gold_lideres`), o Jev responde a **duas perguntas na mesma requisição** (*fan-out*):

| id | tipo | pergunta |
|---|---|---|
| `natureza` | **Choice** | Que tipo de condição é `cid.descricao`? |
| `evitavel` | **Noul** | Mortes e internações por essa condição são em grande parte evitáveis com atenção primária, prevenção ou diagnóstico precoce? |

**Antes de rodar**, rotule à mão a `natureza` de 10 categorias. O Jev é instrumento de medição, não oráculo: confiança baixa é conteúdo de aula, não erro.

**Chave da API:** nunca no notebook. No Free Edition, use um *secret scope* (`databricks secrets create-scope typesafe` e `databricks secrets put-secret typesafe api_key`).

In [ ]:
# As perguntas valem para os dois caminhos (4.6 e 4.7): rode esta célula em qualquer caso.
NATUREZA_INSTR = "Que tipo de condição de saúde descreve a categoria `cid.codigo` da CID-10, `cid.descricao`?"
NATUREZA_CRITERIA = {
    "cronica": "Doença crônica não transmissível: cardiovascular, câncer, diabetes, respiratória crônica, renal crônica",
    "infecciosa": "Doença infecciosa ou parasitária, aguda ou crônica, incluindo pneumonias e infecções virais",
    "gravidez_parto": "Gravidez, parto, puerpério ou condição do recém-nascido",
    "externa": "Lesão, envenenamento, acidente, agressão ou outra causa externa",
    "mal_definida": "Sintoma, sinal ou causa mal definida, sem diagnóstico de doença",
    "contato_servico": "Procedimento, cuidado de saúde ou motivo de contato com o serviço, não uma doença",
}
EVITAVEL_INSTR = ("Mortes e internações por `cid.descricao` são em grande parte evitáveis no Brasil "
                  "com atenção primária, prevenção ou diagnóstico precoce?")

In [ ]:
from typesafe_sdk import TypeSafeClient, Choice, Noul

def chave_typesafe():
    try:
        return dbutils.secrets.get("typesafe", "api_key")
    except Exception:
        return os.environ.get("TYPESAFE_API_KEY")

PERGUNTAS = {
    "natureza": Choice(instructions=NATUREZA_INSTR, criteria=NATUREZA_CRITERIA),
    "evitavel": Noul(instructions=EVITAVEL_INSTR),
}

categorias = (spark.table(T("gold_lideres")).select("cid_cat").distinct()
              .join(dim_cid, "cid_cat").select("cid_cat", "descricao", "capitulo", "observacao").toPandas())
chave = chave_typesafe()
if not chave:
    print("⚠️  Sem chave da TypeSafe (secret scope `typesafe`, chave `api_key`): a seção 4.6 é opcional e foi pulada.")
else:
    resultados = []
    with TypeSafeClient(api_key=chave, model="jev-latest") as client:   # em produção: pinar a versão
        for c in categorias.itertuples():
            state = {"cid": {"codigo": c.cid_cat, "descricao": c.descricao,
                             **({"observacao": c.observacao} if c.observacao else {})}}
            r = client.system_one(state=state, questions=PERGUNTAS)
            n, e = r.choices["natureza"], r.nouls["evitavel"]
            resultados.append({"cid_cat": c.cid_cat, "natureza": n.choice, "natureza_conf": n.confidence,
                               "natureza_probs": json.dumps(n.probabilities), "evitavel": e.noul, "modelo": r.model})
    spark.createDataFrame(pd.DataFrame(resultados)).write.mode("overwrite") \
         .option("overwriteSchema", True).saveAsTable(T("silver_cid_jev"))

### Composição em código: o novo atributo e a comparação com a regra

- `natureza` entra na `dim_cid` só com confiança ≥ 0,6; abaixo disso, `indefinido` (revisão humana).
- `evitavel` vira `sim` (≥ 0,7), `não` (≤ 0,3) ou `incerto`.
- **Divergência com a regra:** o Jev diz `cronica` ou `infecciosa`, mas o capítulo está em `CAPITULOS_FORA`, ou o contrário. Cada divergência é um caso para discutir.

O novo atributo entra na dimensão **existente** com `ALTER TABLE … ADD COLUMNS` e `MERGE INTO`, sem recriar a tabela. Esta é a operação pedida na A2. Como sobrescrevemos o valor, é um **SCD tipo 1**.

In [ ]:
if not spark.catalog.tableExists(T("silver_cid_jev")):
    print("– 4.6 não rodou (sem chave da TypeSafe): nada a compor na dim_cid.")
else:
    (spark.table(T("silver_cid_jev"))
         .select("cid_cat",
                 F.when(F.col("natureza_conf") >= 0.6, F.col("natureza")).otherwise(F.lit("indefinido")).alias("natureza"),
                 F.when(F.col("evitavel") >= 0.7, "sim").when(F.col("evitavel") <= 0.3, "não").otherwise("incerto").alias("evitavel"))
         .createOrReplaceTempView("atributos_jev"))

    colunas_dim = spark.table(T("dim_cid")).columns
    if "natureza" not in colunas_dim:
        spark.sql(f"ALTER TABLE {T('dim_cid')} ADD COLUMNS (natureza STRING, evitavel STRING)")
    spark.sql(f'''MERGE INTO {T("dim_cid")} d USING atributos_jev j ON d.cid_cat = j.cid_cat
                  WHEN MATCHED THEN UPDATE SET d.natureza = j.natureza, d.evitavel = j.evitavel''')

    regra_exclui = F.coalesce(F.col("capitulo").isin(list(CAPITULOS_FORA)), F.lit(False))
    jev_exclui = ~F.col("natureza").isin("cronica", "infecciosa", "indefinido")
    display(spark.table(T("dim_cid")).where(F.col("natureza").isNotNull())
            .withColumn("divergencia", regra_exclui != jev_exclui)
            .select("cid_cat", "descricao", "capitulo", "natureza", "evitavel", "divergencia")
            .orderBy(F.desc("divergencia"), "cid_cat"))

## 4.7 (Opcional) O mesmo julgamento em SQL com `ai_decide`

O Databricks tem uma função SQL nativa, **`ai_decide(state, questions)`**, movida por um modelo de decisão da TypeSafe. O JSON de `questions` é o **mesmo formato** do System One: `type`, `instructions` e `criteria`. Não há chave para guardar, e o dado não sai do workspace.

- `state`: aqui, um JSON por linha montado com `to_json(named_struct(...))`.
- Resposta: um `VARIANT` com `response.answers.<id>`. Atenção aos nomes: no `noul`, o valor vem em **`probability`**; no `choice`, em `choice`, `probabilities` e `confidence`.

A função está em **Beta** e só existe em algumas regiões. Se o seu workspace não a tiver, a célula avisa e segue. Basta ler o SQL.

In [ ]:
import json

QUESTOES = json.dumps({
    "natureza": {"type": "choice", "instructions": NATUREZA_INSTR, "criteria": NATUREZA_CRITERIA},
    "evitavel": {"type": "noul", "instructions": EVITAVEL_INSTR},
}, ensure_ascii=False).replace("'", "\\'")       # vira uma constante SQL entre aspas simples

try:
    decisoes = spark.sql(f'''
        SELECT cid_cat, descricao,
               ai_decide(to_json(named_struct('cid', named_struct('codigo', cid_cat, 'descricao', descricao,
                                                                  'observacao', observacao))),
                         '{QUESTOES}') AS d
        FROM {T("dim_cid")}
        WHERE cid_cat IN (SELECT cid_cat FROM {T("gold_lideres")} WHERE pos = 1)
        LIMIT 10''')
    decisoes = decisoes.selectExpr(
        "cid_cat", "descricao",
        "d:response.answers.natureza.choice::string        AS natureza_sql",
        "round(d:response.answers.natureza.confidence::double, 2) AS confianca_sql",
        "round(d:response.answers.evitavel.probability::double, 2) AS evitavel_sql",
        "d:error_message::string AS erro")
    if spark.catalog.tableExists(T("silver_cid_jev")):            # fez o 4.6? compare os dois
        decisoes = decisoes.join(spark.table(T("silver_cid_jev"))
                                 .select("cid_cat", F.col("natureza").alias("natureza_jev"),
                                         F.round("evitavel", 2).alias("evitavel_jev")), "cid_cat", "left")
    display(decisoes)
except Exception as e:
    print("ai_decide indisponível neste workspace (Beta, depende da região):", str(e).splitlines()[0][:300])

**Para discutir:** o `ai_decide` e o Jev do 4.6 concordam? Onde divergem, a confiança é baixa? Qual dos dois caminhos você usaria **dentro de um pipeline agendado** do Databricks, e qual fora dele? O que muda em **governança** (onde o dado trafega, quem guarda a chave)?

---
### Trilhas — Etapa 4

- **Prático:** recalcule as internações de 2019 usando só `mes == 1` e depois só `mes == 7` (a coluna `mes` está em `silver_internacoes`). A líder muda em quantas UFs? O que isso diz sobre amostrar meses?
- **Criativo:** escreva a **manchete** do seu achado (até 12 palavras) e o lide (2 frases), sustentados pelo mapa da 4.3. Exemplo de tom: *"Por dois anos, um vírus tirou o infarto do topo em quase todo o país"*.
- **Abstrato:** a resposta muda com o **nível** da CID (4.5). Prove com um exemplo numérico pequeno (3 categorias, 2 grupos) que a categoria líder **pode não pertencer** ao grupo líder. Que nível você defenderia para um gestor? E se a pergunta fosse sobre CID-9 (1979–1995): o que precisaria existir para estender a série?

---
# 5. Entrega (individual)

Exporte este notebook executado no modo **`completo`** (**File → Export → IPython Notebook**) e entregue no D2L, junto com um texto de até 1 página contendo:

1. **A resposta do desafio** (TODO 4.3), com os dois mapas da seção 4.3.
2. **Dicionário de dados** do SIH (TODO 1.1) e as **4 decisões de modelagem** que você tomou: nível da CID, capítulos excluídos, UF de residência × de atendimento, meses do SIH.
3. **Robustez**: a tabela da 4.5 e uma frase interpretando-a.
4. **Governança**: o percentual de células com *n* < *k* (3.6) e como você evitou publicá-las.
5. **Trilhas**: quais atividades você fez em cada etapa.
6. (Opcional) A tabela de divergências regra × Jev (4.6) e um caso comentado; se rodou o 4.7, uma frase comparando Jev × `ai_decide`.

### Checklist de autoavaliação

- [ ] As tabelas `bronze_*`, `silver_*`, `dim_cid` e `gold_*` existem no schema `oficina_sus`.
- [ ] Rodar o notebook duas vezes não baixa nem duplica nada (idempotência).
- [ ] Não li nenhum arquivo `DOBR…` e não confundi o prefixo de arquivo `RJ` do SIH com o estado.
- [ ] Contei só AIH com `IDENT = 1`.
- [ ] Sei explicar por que a série começa em 1996 (SIM) e 2008 (SIH), e por que 2026 ficou de fora.
- [ ] Sei explicar por que `B34` é COVID-19 nesta base.
- [ ] Nenhuma chave de API aparece no notebook.